<b> Project Overview</b><br>

Business problem draft<br>
Intended user or decision context<br>
Dataset description and source link<br> [x]
List of raw files or tables<br>
Explanation of what one row means in each important table<br>
Initial row and column counts<br>
Data types<br>
Missing values<br>
Duplicate records or duplicate keys<br>
Suspicious, impossible, or confusing values<br>
Initial cleaning plan<br>
Potential target or outcome variable<br>
Early ethics, privacy, governance, or responsible-use concerns<br>


---
## Loading The Data <br>

The initial step in this is project is to load the required libraries and the datasets. 
 
The libraries required are...   
        - pathlib : a file handler that treats paths as objects rather than text strings.   
        - pandas : a library of functions that are used to manipulate dataframes.  
        - numpy : a library of math functions.  

Next, we define the root folder where the project folders and files will be saved.  

The project directory contains the following folders...  
        - The data folder : This folder contains raw data files.  
        - The output folder : This folder contains the final ABT and processed data files.    
        - The report folder : This folder contains html reports, graphs, and analysis.  



In [12]:
# This code loads libraries and sets directors for the project.
#  pathlib is a file handler. treats paths as objects not strings
# pandas brings in df functions
# numpy provides math functions.

#Checking my Python environment 
import sys; print(sys.executable); print(sys.version)

from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

#Set repository root to the current working directory. One step above readme location.
display(REPO_ROOT)

#Create directories for data, outputs, and reports if they don't exist.
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

#set max column display to 100 rows for pandas dataframes.
pd.set_option("display.max_columns", 100)

/usr/bin/python3
3.13.5 (main, Aug 10 2026, 12:06:59) [GCC 14.2.0]


PosixPath('/workspaces/Project-Option-1-Matt-Flores')

---               
## Dataset Review

Now that the data is loaded we can review and profile each table loaded into the directory.  
There are 5 raw data files loaded into the data folder.
1) **movies** : This is the main df because the primary key, ***movieId***, in this df is the project's **unit of analysis**.  

2) **ratings** : This dataset contains almost 100k reviews throughout 1995 to 2016. This data will need to be aggregated to be merged with the movies table. *Each row in this dataset represents a single user's rating for a movie. There are multiple ratings per movie.*  

3) **links** : Contains the imdbId & tmdbId associated with each movieId. It allows the ratings df to be joined with the credits and keywords tables without having to cross refence the movies dataset, which contains all 3 id types. *Each row in this dataset represents a movieId and related links. Each movieId should have 2 additional Ids.*  

4) **credits** : Lists the directors and top billed actors in each film. Maybe there is a correlation between revenue and directors/actors. *Each row in this dataset represents a movieID with associated directions/actors.*  

5) **keywords** : Contains The Movie Database ID along with a multi-valued field listing descriptive words for each movie. Genre is a broad categorization of a film. keywords df can provide more detail on themes, locations, movie plots, characters, etc. *Each row in this dataset represents a single movie.*

In [13]:
#This code loads the raw data into dataframes. And creates a column summary by table df.
# Load the datasets
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

# Create a dictionary (keys paired with values) to hold the DataFrames for easier access. We can loop through tables, etc.
tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

print("Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.")

Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.


---               
## Data Structure

The first step in data profiling is reviewing the data structure. Data stucture consists of rows and columns. To analyze data we must first understand what each row of the data represents and the attributes (columns) grouped with each row.

Instead of running shape() for each table I thought it would be best to aggregate each table's structure into summary table.  

I appended table name along with the associated columns into a single dataframe.  
I repeated the process a second time to gather the shape info from each table.

In [14]:
# Create a blank list called column_summary that we can populate.
column_summary = []

# Populate the list using a loop that appends table names and each column within that table.
for name, df in tables.items():
    column_summary.append({"table_name": name, **{f"col_{i+1}": col for i, col in enumerate(df.columns)}})

# Build a summary DataFrame that shows the table name and each column name in separate columns.
# Converted summary_data list into a pd.dataframe and set index equal to table name.
column_summary_df = pd.DataFrame(column_summary).set_index("table_name")

# Display the result
display(column_summary_df)

raw_data_shape = []
# Print out each df and it's shape. For is a loop function. "for every name in tables dictionary, print the shape variables"
for name, df in tables.items():
     raw_data_shape.append({"table_name": name, "rows": df.shape[0], "columns": df.shape[1]})

raw_data_shape_df = pd.DataFrame(raw_data_shape).set_index("table_name")

# Display the results
display(raw_data_shape_df)

,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_20,col_21
table_name,,,,,,,,,,,,,,,,,,,,,
movies,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
ratings,userId,movieId,rating,timestamp,rating_datetime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
links,movieId,imdbId,tmdbId,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credits,tmdbId,director,top_cast,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keywords,tmdbId,keywords_list,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,rows,columns
table_name,,
movies,9082,21
ratings,99810,5
links,9125,3
credits,9082,3
keywords,9082,2


We can confirm a few facts:  
1) **Granularity Match**: The movies, credits, and keywords datasets contain 9,082 rows. The links df contains 43 more rows. It's possible the links table contains additional movieids that are not included in the other tables. There may also be duplicate ids in each table, but it seems these df's row level represents a single movie.  

2) **Ratings Shape**: The ratings df has almost 100k rows. The most popular movies might receive a large portion of these reviews. This will require further analysis.  

3) **Aggregation Required**: Since the ratings row level doesn't match the other tables any joining of the ratings table will require aggregation. We could examine the average rating per movie, distribution of review scores, count of reviews, etc.  

4) **Movie Sales Figures**: The movies dataset contains budget and revenue columns. These will be useful when calculating ROI per movie and sorting by revenue. One thing to consider is highly rated movies might not make the most revenue. Many classic movies such as "Fight Club" and "Donnie Darko" had a low or negative ROI.

---               
## Initial Inspection

This step is exploratory. We know the data set structure, but it's helpful to see a snippet of each table. Let's loop through each table and output head(3) and tail(3)

In [36]:
for name, df in tables.items():
    print(f'\n{name}: head and tails')
    display(pd.concat([df.head(3),df.tail(3)]))


movies: head and tails


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995
9079,162672,402672,tt3859980,Mohenjo Daro,Mohenjo Daro,hi,2016-08-11,155.0,15050000,16180000.0,1.423358,6.7,26.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...","[{'name': 'UTV Motion Pictures', 'id': 2320}, ...","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Adventure|Drama|History|Romance,Adventure,2016
9080,163056,315011,tt4262980,Shin Godzilla,シン・ゴジラ,ja,2016-07-29,120.0,15000000,77000000.0,9.285519,6.6,152.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Cine Bazar', 'id': 5896}, {'name': ...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",False,Released,Action|Adventure|Drama|Horror|Science Fiction,Action,2016
9081,163949,391698,tt2531318,The Beatles: Eight Days a Week - The Touring Y...,The Beatles: Eight Days a Week - The Touring Y...,en,2016-09-15,99.0,0,0.0,7.078301,7.6,92.0,"[{'id': 99, 'name': 'Documentary'}, {'id': 104...","[{'name': 'Imagine Entertainment', 'id': 23}, ...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Documentary|Music,Documentary,2016



ratings: head and tails


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02
99807,671,6365,4.0,1070940363,2003-12-09 03:26:03
99808,671,6385,2.5,1070979663,2003-12-09 14:21:03
99809,671,6565,3.5,1074784724,2004-01-22 15:18:44



links: head and tails


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
9122,163949,2531318,391698.0
9123,164977,27660,137608.0
9124,164979,3447228,410803.0



credits: head and tails


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret
9079,402672,Ashutosh Gowariker,Hrithik Roshan|Pooja Hegde|Kabir Bedi
9080,315011,Hideaki Anno,Hiroki Hasegawa|Yutaka Takenouchi|Satomi Ishihara
9081,391698,Ron Howard,Paul McCartney|Ringo Starr|John Lennon



keywords: head and tails


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men
9079,402672,bollywood
9080,315011,monster|godzilla|giant monster|destruction|kai...
9081,391698,music|documentary


The head and tail output resulted in some key findings:
    - genres, production_companies, and production_countries are lists.
    - genre list is genres reformatted as a string list delimited by "|"
    - I'm not sure yet how ratings[timestamp] will be used. It may be redundant for this analysis.
    - top_cast is a string list delimited by "|"
    - keywords_list appears very detailed. Also a string seperated by "|"

---               
## Inspecting Missing, Duplicate, & Unusual Values

Next, we will profile each data set and document issues that may require cleaning in the next project milestone. There are 5 sections, each dedicated to a dataset. 

For each dataset we will:  
    - Review data types.  
    - Check for missing values, empty strings, and zero values.  
    - Ensure no primary key is duplicated.  
    - Note suspicious, impossible, or confusing values


---
**Movies**

The movies dataset contains over 9000 movies with 21 columns of details. We will profile the data below using python functions such as .dtypes, nunique, .isna.mean, and .isna.sum. It's more clear if we can view all the details regarding a dataset in a single table output.

In [19]:
# Create a dataframe to store movies data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# Convert all number columns into true or false so we can count where 0 is true
num_cols = movies.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = movies.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
movies_profile = pd.DataFrame({
    'dtype': movies.dtypes, 
    'unique_values' : movies.nunique(),
    'duplicate_values' : movies.shape[0] - movies.nunique(),
    'missing_rate': round(movies.isna().mean() * 100,2), 
    'missing_count': movies.isna().sum(),
    'zero_count': zero_mask.sum().reindex(movies.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(movies.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(movies.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(movies.columns, fill_value=0) * 100,2)
})

print("Movies Profile Table")
display(movies_profile.sort_values(by='missing_rate', ascending=False))


Movies Profile Table


,dtype,unique_values,duplicate_values,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
genres_list,object,1694,7388,0.39,35,0,0.00,0,0.0
primary_genre,object,20,9062,0.39,35,0,0.00,0,0.0
status,object,4,9078,0.02,2,0,0.00,0,0.0
tmdbId,int64,9082,0,0.00,0,0,0.00,0,0
movieId,int64,9082,0,0.00,0,0,0.00,0,0
original_title,object,8841,241,0.00,0,0,0.00,0,0.0
title,object,8809,273,0.00,0,0,0.00,0,0.0
imdb_id,object,9082,0,0.00,0,0,0.00,0,0.0
original_language,object,42,9040,0.00,0,0,0.00,0,0.0
revenue,float64,4555,4527,0.00,0,4322,47.59,0,0


The output table generated above provides some key profiling information:

1) **dtypes** : There are a few things that could be changed.
    - ***release_date*** is an object. It should be converted to a date using pd.to_datetime().
    - ***vote_count*** is set as float64 but it's a whole number so it should be int64.
    - ***revenue*** and ***budget*** represent dollars but are different dtypes. Let's plan on standardizing them as float64. Float64 allows for NaN values, which will prevent them from being included in math calculations.
    - (***primary_genre, status, original_language***) contain few unqiue values. They could be converted to ***category*** data type. Doing so will improve grouping and sorting processing speeds.  

2) **Missing Values** : The output table display missing count/rate, zero count/rate, and empty count/rate.
    - Missing count/rate are calculated using isna mean and isna sum. Those functions identify null or missing values. The data appears to be extremely *complete* in regards to null and missing.
    - The zero count/rate reveal missing information that will impact the analysis. *Almost 50% of the revenue/budget values are 0*. This means when conducting analysis on revenue/budget we must omit these movieIds. Including them in any equations will skew the results.
    - empty count/rate shows 0 because there are no blank strings or values. There are only nulls/isna or 0.

3) **Duplicates** : The movies dataset has 9,082 rows.
    - ***movieId, tmdbId, and imdb_id*** have 9,082 rows. This makes logic sense and proves that the row level data represents a single movie.
    - It's possible to have duplicate movie ***titles***. The data spans from 1902 to 2016. Over almost 100 years there have been 273 movies with a title that's already been used.
    - Any column that represents a group or label is allowed to have duplicates. These are: ***grenes_list, primary_genre, status, original language, genres, adult, production countries, production companies, and release year***.
    - It's suspicious that ***revenue*** and ***budget*** have duplicates, but remember about 50% of them are 0. So 50% of them should be duplicates. Also consider that the dollars amounts have low granularity. The budget/revenue figures are a rough ball park of the real amounts. Which reduces the varitey of the data.
    - ***vote_average*** is based on a scale of 1-10. It makes sense that there are duplicates.

**Movies head and tail display**



In [16]:
##### SCRATCH NOTES ############

movies["genres"].head(20)
#(movies['revenue'] % 1 > 0).sum()
#display(movies["genres"].unique())
#display(movies)
display(movies[["genres", "genres_list"]][movies['title'] == 'Toy Story'])

#movies[movies['original_language'] != 'en'].sort_values(by='revenue', ascending=False)

print(movies.loc[movies['title'] == 'Toy Story', 'genres'].iloc[0])

# Find rows where genres and genres_list are not equal
mismatches = movies[movies['genres'] != movies['genres_list']]

# Display the title and both columns for the first few mismatches
display(mismatches[['title', 'genres', 'genres_list']].head(10))

import ast

# Safely convert the stringified list of dicts into a real Python list of dicts
sample_val = ast.literal_eval(movies['genres'].iloc[0])

# Extract just the names
genre_names = [g['name'] for g in sample_val]
print(genre_names)  # Output: ['Animation', 'Comedy', 'Family']

import ast

# Function to parse the dict string and return a pipe-separated string of names
def parse_genre_names(val):
    if pd.isna(val):
        return ""
    try:
        # Convert stringified list of dicts to real list, extract names, join with '|'
        items = ast.literal_eval(val)
        return "|".join([item['name'] for item in items])
    except:
        return ""

# Create a temporary test column from 'genres'
test_extracted = movies['genres'].apply(parse_genre_names)

# Check if they match 100%
matches = (test_extracted == movies['genres_list'])
print(f"Do they match completely? {matches.all()}")

# If False, see how many rows differ
print(f"Number of mismatches: {(~matches).sum()}")

# Create the test_extracted column if you haven't already
# test_extracted = movies['genres'].apply(parse_genre_names)

# Filter for the mismatches and look at them side-by-side
mismatched_rows = movies[test_extracted != movies['genres_list']]

# Display the title, raw dictionary string, and genres_list for a few of them
display(mismatched_rows[['title', 'genres', 'genres_list']].head(10))

,genres,genres_list
0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",Animation|Comedy|Family


[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]


,title,genres,genres_list
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",Animation|Comedy|Family
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Adventure|Fantasy|Family
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",Romance|Comedy
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",Comedy|Drama|Romance
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Comedy
5,Heat,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...",Action|Crime|Drama|Thriller
6,Sabrina,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...",Comedy|Romance
7,Tom and Huck,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action|Adventure|Drama|Family
8,Sudden Death,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action|Adventure|Thriller
9,GoldenEye,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",Adventure|Action|Thriller


['Animation', 'Comedy', 'Family']
Do they match completely? False
Number of mismatches: 106


,title,genres,genres_list
78,Last Summer in the Hamptons,[],NaN
139,Congo,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action|Adventure|Drama|Mystery|Science Fiction
459,Radioland Murders,"[{'id': 35, 'name': 'Comedy'}, {'id': 80, 'nam...",Comedy|Crime|Drama|Music|Mystery
521,Aladdin,"[{'id': 16, 'name': 'Animation'}, {'id': 10751...",Animation|Family|Comedy|Adventure|Fantasy
650,My Life and Times With Antonin Artaud,[],NaN
683,The Story of Xinghua,[],NaN
684,The Day the Sun Turned Cold,[],NaN
695,Small Faces,[],NaN
710,1-900,[],NaN
784,Beat the Devil,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action|Adventure|Comedy|Crime|Drama


In [21]:
movies.describe().T

#dup_titles = movies[movies['title'].duplicated(keep=False)]
#display(dup_titles.sort_values('title')[['movieId', 'title', 'release_date']].head(10))


,count,mean,std,min,25%,50%,75%,max
movieId,9082.0,3.096374e+04,4.065969e+04,1.000000,2843.250000,6265.500000,5.602625e+04,1.639490e+05
tmdbId,9082.0,3.871046e+04,6.212341e+04,2.000000,9446.250000,15775.000000,3.901025e+04,4.164370e+05
runtime,9082.0,1.056614e+02,3.035071e+01,0.000000,93.000000,102.000000,1.150000e+02,1.140000e+03
budget,9082.0,1.664789e+07,3.345276e+07,0.000000,0.000000,100000.000000,2.000000e+07,3.800000e+08
revenue,9082.0,4.918728e+07,1.301801e+08,0.000000,0.000000,271608.000000,3.672528e+07,2.787965e+09
popularity,9082.0,7.268905e+00,9.372314e+00,0.000004,2.954521,6.418871,9.861742e+00,5.474883e+02
vote_average,9082.0,6.362178e+00,1.038748e+00,0.000000,5.800000,6.500000,7.000000e+00,1.000000e+01
vote_count,9082.0,4.390509e+02,9.961565e+02,0.000000,29.000000,96.000000,3.657500e+02,1.407500e+04
release_year,9082.0,1.991952e+03,1.938771e+01,1902.000000,1984.000000,1997.000000,2.006000e+03,2.016000e+03


In [30]:
# Populate the list using a loop that appends table names and each column within that table.
#for name, df in tables.items():
#    column_summary.append({"table_name": name, **{f"col_{i+1}": col for i, col in enumerate(df.columns)}})


movie_frequency_columns = movies[['budget', 'revenue', 'release_date']]

# Loop through only the selected subset of columns
for col in movies.columns:
    print(f"--- Value counts for: {col} ---")
    display(movies[col].value_counts().head(10))

--- Value counts for: movieId ---


movieId
163949    1
1         1
2         1
3         1
4         1
5         1
6         1
7         1
8         1
9         1
Name: count, dtype: int64

--- Value counts for: tmdbId ---


tmdbId
391698    1
862       1
8844      1
15602     1
31357     1
11862     1
949       1
11860     1
45325     1
9091      1
Name: count, dtype: int64

--- Value counts for: imdb_id ---


...

genres_list
Drama                   732
Comedy                  554
Drama|Romance           375
Comedy|Drama            314
Documentary             268
Comedy|Romance          266
Comedy|Drama|Romance    245
Horror                  140
Horror|Thriller         116
Drama|Thriller          112
Name: count, dtype: int64

--- Value counts for: primary_genre ---


primary_genre
Drama          2413
Comedy         2168
Action         1177
Adventure       517
Horror          497
Crime           417
Documentary     383
Thriller        263
Fantasy         244
Animation       236
Name: count, dtype: int64

--- Value counts for: release_year ---


release_year
2000    273
1998    272
1996    271
2002    268
1999    266
1995    266
2001    263
1997    262
2006    259
2007    254
Name: count, dtype: int64

In [22]:
# Create a dataframe to store movies data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# 1. Numeric masks for zeros
num_cols = movies.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# 2. Text masks for empty/whitespace strings
text_cols = movies.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
movies_datatypes = pd.DataFrame({
    'dtype': movies.dtypes, 
    'missing_rate': round(movies.isna().mean() * 100,2), 
    'missing_count': movies.isna().sum(),
    
    'zero_count': zero_mask.sum().reindex(movies.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(movies.columns, fill_value=0) * 100,2),
    
    'empty_count': empty_mask.sum().reindex(movies.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(movies.columns, fill_value=0) * 100,2)
})

print("--- movies Dtypes and Missing Review ---")
display(movies_datatypes.sort_values(by='missing_rate', ascending=False))

# Create a movies missing dataset that lists every movieId with a missing value.
# This will be used later to filter out missing movies from analysis. I could query "not in movies_missing[movieId]"
# I could also make movies_copy = movies - movies_missing['movieID']
movies_missing = movies[movies.isna().any(axis=1)][['movieId','title','release_date','genres_list','primary_genre','status']]
display(movies_missing.sort_values(by = 'status', ascending = True).head())

# This will return the exact count of movies with a budget of 0
zero_budget_count = (movies['budget'] == 0).sum()
zero_revenue_count = (movies['revenue'] == 0).sum()

display(movies[movies['revenue'] == 0])

display(movies['vote_average'].value_counts())
display(movies.sort_values(by='popularity', ascending = False).head(20))

print(f"zero budget count: {zero_budget_count}\nzero revenue count: {zero_revenue_count}")

--- movies Dtypes and Missing Review ---


,dtype,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
genres_list,object,0.39,35,0,0.00,0,0.0
primary_genre,object,0.39,35,0,0.00,0,0.0
status,object,0.02,2,0,0.00,0,0.0
tmdbId,int64,0.00,0,0,0.00,0,0
movieId,int64,0.00,0,0,0.00,0,0
original_title,object,0.00,0,0,0.00,0,0.0
title,object,0.00,0,0,0.00,0,0.0
imdb_id,object,0.00,0,0,0.00,0,0.0
original_language,object,0.00,0,0,0.00,0,0.0
revenue,float64,0.00,0,4322,47.59,0,0


,movieId,title,release_date,genres_list,primary_genre,status
78,84,Last Summer in the Hamptons,1995-11-22,NaN,NaN,Released
5117,7358,Searching for Debra Winger,2002-07-13,NaN,NaN,Released
5252,7872,Getting It Right,1989-05-05,NaN,NaN,Released
5328,8131,Pursuit of Happiness,2001-01-17,NaN,NaN,Released
5470,8622,Fahrenheit 9/11,2004-06-25,NaN,NaN,Released


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995
6,7,11860,tt0114319,Sabrina,Sabrina,en,1995-12-15,127.0,58000000,0.0,6.677277,6.2,141.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'is...",False,Released,Comedy|Romance,Comedy,1995
7,8,45325,tt0112302,Tom and Huck,Tom and Huck,en,1995-12-22,97.0,0,0.0,2.561161,5.4,45.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Walt Disney Pictures', 'id': 2}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Drama|Family,Action,1995
11,12,12110,tt0112896,Dracula: Dead and Loving It,Dracula: Dead and Loving It,en,1995-12-22,88.0,0,0.0,5.430331,5.7,210.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[{'name': 'Columbia Pictures', 'id': 5}, {'nam...","[{'iso_3166_1': 'FR', 'name': 'France'}, {'iso...",False,Released,Comedy|Horror,Comedy,1995
21,22,1710,tt0112722,Copycat,Copycat,en,1995-10-27,124.0,0,0.0,10.701801,6.5,199.0,"[{'id': 18, 'name': 'Drama'}, {'id': 53, 'name...","[{'name': 'Regency Enterprises', 'id': 508}, {...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Drama|Thriller,Drama,1995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9075,161830,314420,tt3732950,Body,Body,en,2015-01-25,75.0,0,0.0,6.236714,5.0,16.0,"[{'id': 18, 'name': 'Drama'}, {'id': 53, 'name...","[{'name': 'Oscilloscope Laboratories', 'id': 4...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Drama|Thriller,Drama,2015
9076,161918,390989,tt4831420,Sharknado 4: The 4th Awakens,Sharknado 4: The 4th Awakens,en,2016-07-31,85.0,0,0.0,4.574494,4.3,88.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[{'name': 'The Asylum', 'id': 1311}, {'name': ...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Horror|Science Fiction,Comedy,2016
9077,161944,159550,tt0255313,The Last Brickmaker in America,The Last Brickmaker in America,en,2001-09-23,85.0,8000000,0.0,0.038998,7.0,1.0,"[{'id': 18, 'name': 'Drama'}]","[{'name': 'Nasser Entertainment', 'id': 35802}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Drama,Drama,2001
9078,162542,392572,tt5165344,Rustom,रुस्तम,hi,2016-08-12,150.0,1000000,0.0,7.333139,7.3,25.0,"[{'id': 53, 'name': 'Thriller'}, {'id': 10749,...","[{'name': 'KriArj Entertainment', 'id': 91689}]","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Thriller|Romance,Thriller,2016


vote_average
6.4    417
6.7    405
6.5    401
6.6    398
7.0    383
      ... 
2.7      1
1.5      1
1.2      1
1.9      1
9.2      1
Name: count, Length: 72, dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
8889,135887,211672,tt2293640,Minions,Minions,en,2015-06-17,91.0,74000000,1.156731e+09,547.488298,6.4,4729.0,"[{'id': 10751, 'name': 'Family'}, {'id': 16, '...","[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Family|Animation|Adventure|Comedy,Family,2015
8664,115617,177572,tt2245084,Big Hero 6,Big Hero 6,en,2014-10-24,102.0,165000000,6.521054e+08,213.849907,7.8,6289.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 10751...","[{'name': 'Walt Disney Pictures', 'id': 2}, {'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Family|Animation|Action|Comedy,Adventure,2014
8752,122904,293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108.0,58000000,7.831130e+08,187.860492,7.4,11444.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Comedy,Action,2016
7390,72998,19995,tt0499549,Avatar,Avatar,en,2009-12-10,162.0,237000000,2.787965e+09,185.070892,7.2,12114.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Ingenious Film Partners', 'id': 289...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Fantasy|Science Fiction,Action,2009
8654,115149,245891,tt2911666,John Wick,John Wick,en,2014-10-22,101.0,20000000,8.876166e+07,183.870374,7.0,5499.0,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...","[{'name': 'Summit Entertainment', 'id': 491}, ...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",False,Released,Action|Thriller,Action,2014
8586,112556,210577,tt2267998,Gone Girl,Gone Girl,en,2014-10-01,145.0,61000000,3.693304e+08,154.801009,7.9,6023.0,"[{'id': 9648, 'name': 'Mystery'}, {'id': 53, '...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Mystery|Thriller|Drama,Mystery,2014
8683,116823,131631,tt1951265,The Hunger Games: Mockingjay - Part 1,The Hunger Games: Mockingjay - Part 1,en,2014-11-18,123.0,125000000,7.521002e+08,147.098006,6.6,5767.0,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...","[{'name': 'Lionsgate', 'id': 1632}, {'name': '...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Science Fiction|Adventure|Thriller,Science Fiction,2014
8753,122920,271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147.0,250000000,1.153304e+09,145.882135,7.1,7462.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'name': 'Studio Babelsberg', 'id': 264}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Action|Science Fiction,Adventure,2016
266,296,680,tt0110912,Pulp Fiction,Pulp Fiction,en,1994-09-10,154.0,8000000,2.139288e+08,140.950236,8.3,8670.0,"[{'id': 53, 'name': 'Thriller'}, {'id': 80, 'n...","[{'name': 'Miramax Films', 'id': 14}, {'name':...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Thriller|Crime,Thriller,1994
6897,58559,155,tt0468569,The Dark Knight,The Dark Knight,en,2008-07-16,152.0,185000000,1.004558e+09,123.167259,8.3,12269.0,"[{'id': 18, 'name': 'Drama'}, {'id': 28, 'name...","[{'name': 'DC Comics', 'id': 429}, {'name': 'L...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Drama|Action|Crime|Thriller,Drama,2008


zero budget count: 4443
zero revenue count: 4322


In [54]:
## In this code I examine the movies revenue. This project focuses on the primary unit of data, which is movies. This table is most important.
# limit column width so the text doesnt wrap and make the rows wide.
pd.set_option('display.max_colwidth',12)

# Drop only the columns that cause wide text wrapping
movie_cols_to_show = [col for col in movies.columns if col not in ["genres", "production_companies", "production_countries"]]
#display(movie_cols_to_show)

# Display top 3 movies by revenue
display(
    movies.sort_values(by="revenue", ascending=True).head(3)
)

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
3017,3780,37744,tt0042897,Rocketsh...,Rocketsh...,en,1950-06-02,77.0,0,0.0,1.113762,5.1,15.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Science ...,Science ...,1950
3044,3813,15867,tt0077742,Interiors,Interiors,en,1978-08-02,93.0,0,0.0,4.515797,6.7,92.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Drama,Drama,1978
3046,3816,29263,tt0089276,The Offi...,La histo...,es,1985-04-03,112.0,0,0.0,1.956906,7.3,24.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Drama|Hi...,Drama,1985


In [55]:
# Display top 3 movies by release date desc. Release date is string, but sorting still works on it bc it's in numerical order.
display(movies.sort_values(by="release_date", ascending=False)[["movieId", "title", "release_date"]].head(3))

# Convert release_date to actual datetime objects, then sort
movies_sorted = movies.copy()
movies_sorted['release_date'] = pd.to_datetime(movies_sorted['release_date'], errors='coerce')

display(
    movies_sorted.sort_values(by="release_date", ascending=False)[["movieId", "title", "release_date"]].head(3)
)

#Sorting by date has same result. Most recent movie is september 2023.

,movieId,title,release_date
9023,151307,The Love...,2016-09-23
9081,163949,The Beat...,2016-09-15
8747,122888,Ben-Hur,2016-08-17


,movieId,title,release_date
9023,151307,The Love...,2016-09-23
9081,163949,The Beat...,2016-09-15
8747,122888,Ben-Hur,2016-08-17


In [56]:
display(ratings['movieId'].drop_duplicates())

display(ratings['rating_datetime'].sort_values(ascending=False).iloc[0])
display(ratings['rating_datetime'].sort_values(ascending=True).iloc[0])

display(movies['release_date'].sort_values(ascending=False).iloc[0])
display(movies['release_date'].sort_values(ascending=True).iloc[0])

0           31
1         1029
2         1061
3         1129
4         1172
         ...  
98938    64997
98966    72380
99081      129
99485     4736
99626     6425
Name: movieId, Length: 9025, dtype: int64

Timestamp('2016-10-16 17:57:24')

Timestamp('1995-01-09 11:46:49')

'2016-09-23'

'1902-09-01'

In [57]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,object
title,object
original_title,object
original_language,object
release_date,object
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime...


...

,dtype
tmdbId,int64
director,object
top_cast,object


,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,object


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


In [58]:
for name, df in tables.items():
    print(f"\n--- {name} Profile Summary ---")
    
    # 1. Create a summary DataFrame containing dtypes and missing rates
    summary_df = pd.DataFrame({
        'dtype': df.dtypes,
        'missing_rate': df.isna().mean(),
        'missing_count': df.isna().sum()
    })
    
    # 2. Display the combined table
    display(summary_df.sort_values(by='missing_rate', ascending=False))


--- movies Profile Summary ---


,dtype,missing_rate,missing_count
genres_list,object,0.003854,35
primary_genre,object,0.003854,35
status,object,0.000220,2
tmdbId,int64,0.000000,0
movieId,int64,0.000000,0
original_title,object,0.000000,0
title,object,0.000000,0
imdb_id,object,0.000000,0
original_language,object,0.000000,0
revenue,float64,0.000000,0



--- ratings Profile Summary ---


,dtype,missing_rate,missing_count
userId,int64,0.0,0
movieId,int64,0.0,0
rating,float64,0.0,0
timestamp,int64,0.0,0
rating_datetime,datetime...,0.0,0



--- links Profile Summary ---


,dtype,missing_rate,missing_count
tmdbId,float64,0.001425,13
movieId,int64,0.000000,0
imdbId,int64,0.000000,0



--- credits Profile Summary ---


,dtype,missing_rate,missing_count
top_cast,object,0.009910,90
director,object,0.002643,24
tmdbId,int64,0.000000,0



--- keywords Profile Summary ---


,dtype,missing_rate,missing_count
keywords_list,object,0.083792,761
tmdbId,int64,0.000000,0


In [ ]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Animatio...,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Adventur...,Adventure,1995
2,3,15602,tt0113228,Grumpier...,Grumpier...,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Romance|...,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-1...
1,1,1029,3.0,1260759179,2009-12-1...
2,1,1061,3.0,1260759182,2009-12-1...



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Las...,Tom Hank...
1,8844,Joe John...,Robin Wi...
2,15602,Howard D...,Walter M...



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy...
1,8844,board ga...
2,15602,fishing|...


In [49]:
print("--- Movies ---")
display(movies[movies["movieId"] == 1])

print("--- Ratings ---")
display(ratings[ratings["movieId"] == 1])

print("--- Links ---")
display(links[links["movieId"] == 1])

print("--- Credits ---")
display(credits[credits["tmdbId"] == 862])

print("--- Keywords ---")
display(keywords[keywords["tmdbId"] == 862])

display(movies.sort_values(by="revenue", ascending=False).head(10))

display(movies.sort_values(by="release_date", ascending=False).head(10))

movies[movies.isna().any(axis=1)]

display(movies[movies.isna().any(axis=1) & (movies["revenue"] != 0)])

# Group by movieId to get the average, median, and total rating count
rating_summary = ratings.groupby("movieId").agg(
    rating_avg=("rating", "mean"),
    rating_median=("rating", "median"),
    rating_count=("userId", "count")
).reset_index()

print("--- Merge Rating Summary to Movie df ---")
rating_summary_with_titles = rating_summary.merge(
    movies[["movieId", "title", "vote_average", "vote_count"]], 
    on="movieId", 
    how="left"
)

# Display top 100 with titles
display(rating_summary_with_titles.sort_values(by="rating_count", ascending=False).head(100))

--- Movies ---


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Animatio...,Animation,1995


--- Ratings ---


,userId,movieId,rating,timestamp,rating_datetime
494,7,1,3.0,851866703,1996-12-2...
697,9,1,4.0,938629179,1999-09-2...
887,13,1,5.0,1331380058,2012-03-1...
959,15,1,2.0,997938310,2001-08-1...
3099,19,1,3.0,855190091,1997-02-0...
...,...,...,...,...,...
98341,660,1,2.5,1436680062,2015-07-1...
98523,663,1,4.0,1438397999,2015-08-0...
98549,664,1,3.5,1362421730,2013-03-0...
99664,670,1,4.0,938782344,1999-10-0...


--- Links ---


...

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
7390,72998,19995,tt0499549,Avatar,Avatar,en,2009-12-10,162.0,237000000,2.787965...,185.070892,7.2,12114.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2009
8746,122886,140607,tt2488496,Star War...,Star War...,en,2015-12-15,136.0,245000000,2.068224...,31.626013,7.5,7993.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2015
1355,1721,597,tt0120338,Titanic,Titanic,en,1997-11-18,194.0,200000000,1.845034...,26.889070,7.5,7770.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Drama|Ro...,Drama,1997
7864,89745,24428,tt0848228,The Aven...,The Aven...,en,2012-04-25,143.0,220000000,1.519558...,89.887648,7.4,12000.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Science ...,Science ...,2012
8700,117529,135397,tt0369610,Jurassic...,Jurassic...,en,2015-06-09,124.0,150000000,1.513529...,32.790475,6.5,8842.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2015
8820,130634,168259,tt2820852,Furious 7,Furious 7,en,2015-04-01,137.0,190000000,1.506249...,27.275687,7.3,4253.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action,Action,2015
8749,122892,99861,tt2395427,Avengers...,Avengers...,en,2015-04-22,141.0,280000000,1.405404...,37.379420,7.3,6908.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2015
7816,88125,12445,tt1201607,Harry Po...,Harry Po...,en,2011-07-07,130.0,125000000,1.342000...,24.990737,7.9,6141.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Family|F...,Family,2011
8429,106696,109445,tt2294629,Frozen,Frozen,en,2013-11-27,102.0,150000000,1.274219...,24.248243,7.3,5440.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Animatio...,Animation,2013
8280,102125,68721,tt1300854,Iron Man 3,Iron Man 3,en,2013-04-18,130.0,200000000,1.215440...,23.721243,6.8,8951.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2013


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
9023,151307,373355,tt5278868,The Love...,The Love...,en,2016-09-23,100.0,0,0.0,0.744523,7.0,4.0,[{'id': ...,[],[],False,Released,Documentary,Documentary,2016
9081,163949,391698,tt2531318,The Beat...,The Beat...,en,2016-09-15,99.0,0,0.0,7.078301,7.6,92.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Document...,Documentary,2016
8747,122888,271969,tt2638144,Ben-Hur,Ben-Hur,en,2016-08-17,125.0,100000000,94061311.0,11.510210,5.3,642.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Adventur...,Adventure,2016
9073,161582,338766,tt2582782,Hell or ...,Hell or ...,en,2016-08-12,102.0,12000000,37589296.0,12.565896,7.2,1304.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Crime|Dr...,Crime,2016
9078,162542,392572,tt5165344,Rustom,रुस्तम,hi,2016-08-12,150.0,1000000,0.0,7.333139,7.3,25.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Thriller...,Thriller,2016
9079,162672,402672,tt3859980,Mohenjo ...,Mohenjo ...,hi,2016-08-11,155.0,15050000,16180000.0,1.423358,6.7,26.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Adventur...,Adventure,2016
8885,135536,297761,tt1386697,Suicide ...,Suicide ...,en,2016-08-02,123.0,175000000,745600054.0,42.965027,5.9,7717.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2016
9076,161918,390989,tt4831420,Sharknad...,Sharknad...,en,2016-07-31,85.0,0,0.0,4.574494,4.3,88.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Comedy|H...,Comedy,2016
9080,163056,315011,tt4262980,Shin God...,シン・ゴジラ,ja,2016-07-29,120.0,15000000,77000000.0,9.285519,6.6,152.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|A...,Action,2016
9061,160438,324668,tt4196776,Jason Bo...,Jason Bo...,en,2016-07-27,123.0,120000000,415484914.0,19.133256,5.9,2386.0,[{'id': ...,[{'name'...,[{'iso_3...,False,Released,Action|T...,Action,2016


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
4544,6269,51927,tt0334416,Stevie,Stevie,en,2002-09-09,140.0,0,97000.0,0.489997,6.7,13.0,[{'id': ...,[],[],False,NaN,Document...,Documentary,2002
5470,8622,1777,tt0361596,Fahrenhe...,Fahrenhe...,en,2004-06-25,122.0,6000000,119114517.0,6.839460,6.9,403.0,[],[{'name'...,[{'iso_3...,False,Released,NaN,NaN,2004


--- Merge Rating Summary to Movie df ---


,movieId,rating_avg,rating_median,rating_count,title,vote_average,vote_count
321,356,4.054252,4.0,341,Forrest ...,8.2,8147.0
266,296,4.256173,4.5,324,Pulp Fic...,8.3,8670.0
284,318,4.487138,5.0,311,The Shaw...,8.5,8358.0
525,593,4.138158,4.0,304,The Sile...,8.1,4549.0
232,260,4.221649,4.5,291,Star Wars,8.1,6778.0
...,...,...,...,...,...,...,...
6872,58559,4.235537,4.5,121,The Dark...,8.3,12269.0
1902,2396,3.966942,4.0,121,Shakespe...,6.8,831.0
958,1206,4.000000,4.0,121,A Clockw...,8.0,3432.0
37,39,3.550000,3.5,120,Clueless,6.9,828.0
